In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
from _notebook_init import DATA_PROCESSED

from weather_markov.models.nonuniform_Markov_chain import NonUniformMarkovChainPredictor
from weather_markov.preprocessing.discretizer import TemperatureDiscretizer
from weather_markov.visualization.plots import (
    plot_graph_network,
    plot_prediction_distribution,
    plot_transition_matrix,
)


### test on previous years

In [ ]:
decade_df = pd.read_csv(DATA_PROCESSED / "decades.csv")

disc = TemperatureDiscretizer.from_equal_width(n_bins=10, t_min=-15, t_max=20)

train_years = range(2000, 2024)
test_years = range(2024, 2026)

train_df = decade_df[decade_df["year"].isin(train_years)]
test_df = decade_df[decade_df["year"].isin(test_years)]

disc.fit(train_df["avg_temperature"])
model = NonUniformMarkovChainPredictor(disc).fit(train_df)

len(model.transition_matrices), model.transition_labels[0], model.transition_labels[-1]


In [ ]:
for label, matrix in zip(model.transition_labels, model.transition_matrices):
    print(f"{label[0]} -> {label[1]}: {matrix.shape}")
    # direct heatmap via existing helper expects TransitionGraph; use pandas plot below if needed


In [ ]:
# Optional: visualize first and last step matrices as heatmaps
import seaborn as sns
import matplotlib.pyplot as plt

for i in [0, -1]:
    label = model.transition_labels[i]
    matrix = model.transition_matrices[i]
    sns.heatmap(matrix, annot=False, cmap="YlOrRd")
    plt.title(f"Transition {label[0]} -> {label[1]}")
    plt.show()


In [ ]:
results = []

for test_year in test_years:
    feb_first_decade_temp = test_df[
        (test_df["year"] == test_year)
        & (test_df["month"] == 2)
        & (test_df["decade"] == 1)
    ]["avg_temperature"].iloc[0]

    may_first_decade_temp = test_df[
        (test_df["year"] == test_year)
        & (test_df["month"] == 5)
        & (test_df["decade"] == 1)
    ]["avg_temperature"].iloc[0]

    start_state = disc.transform(pd.Series([feb_first_decade_temp])).iloc[0]
    pred_dist = model.predict(start_state)
    pred_label = model.predict_label(start_state)
    true_label = disc.transform(pd.Series([may_first_decade_temp])).iloc[0]

    results.append(
        {
            "year": test_year,
            "pred": pred_label,
            "true": true_label,
            "correct": pred_label == true_label,
            "dist": pred_dist,
        }
    )


In [ ]:
results_df = pd.DataFrame(results)
results_df

### 2026 prediction

In [ ]:
disc = TemperatureDiscretizer.from_equal_width(n_bins=10, t_min=-20, t_max=25)

train_years = range(2000, 2026)

train_df = decade_df[decade_df["year"].isin(train_years)]

disc.fit(train_df["avg_temperature"])
model = NonUniformMarkovChainPredictor(disc).fit(train_df)


In [ ]:
current_year = 2026

feb_first_decade_temp = decade_df[
    (decade_df["year"] == current_year)
    & (decade_df["month"] == 2)
    & (decade_df["decade"] == 1)
]["avg_temperature"].iloc[0]

start_state = disc.transform(pd.Series([feb_first_decade_temp])).iloc[0]
pred_dist = model.predict(start_state)
pred_label = model.predict_label(start_state)

pred_label


In [ ]:
plot_prediction_distribution(pred_dist)